# Error Analysis — Cross-Encoder v0.6 Ensemble

Mục tiêu: Hiểu model đang fail ở đâu để định hướng v0.7.

**Phân tích:**
1. Confusion matrix theo class
2. Accuracy và bias theo từng class
3. Phân bố error magnitude
4. Systematic bias (model có xu hướng over/under-predict không?)
5. Worst-case examples (những pair bị predict sai nhiều nhất)

In [1]:
# === CELL 1: Setup ===
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.7
!git pull origin experiment/cross-encoder-v0.7
!pip install -q sentence-transformers

import torch
print('CUDA:', torch.cuda.is_available())

Cloning into '/content/Ai-Recruiter-Mini-Ai-Service'...
remote: Enumerating objects: 7420, done.
remote: Counting objects: 100% (412/412), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 7420 (delta 242), reused 219 (delta 180), pack-reused 7008 (from 2)
Receiving objects: 100% (7420/7420), 32.44 MiB | 12.36 MiB/s, done.
Resolving deltas: 100% (4407/4407), done.
/content/Ai-Recruiter-Mini-Ai-Service
Branch 'experiment/cross-encoder-v0.7' set up to track remote branch 'experiment/cross-encoder-v0.7' from 'origin'.
Switched to a new branch 'experiment/cross-encoder-v0.7'
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.7 -> FETCH_HEAD
Already up to date.
CUDA: True


In [2]:
# === CELL 2: Load test data (full records, không chỉ InputExample) ===
import json
import numpy as np
from sentence_transformers import CrossEncoder

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

# Load full records để có true_label, score, cv_text, jd_text
test_records = load_jsonl('datasets/versions/v0.5/cross_encoder/cross_encoder_test.jsonl')

SCORE_RANGES = {
    'poor_match':      (0,  39),
    'weak_match':      (40, 59),
    'moderate_match':  (60, 74),
    'strong_match':    (75, 89),
    'excellent_match': (90, 100),
}
LABELS_ORDER = ['poor_match', 'weak_match', 'moderate_match', 'strong_match', 'excellent_match']

def score_to_label(score):
    for label, (lo, hi) in SCORE_RANGES.items():
        if lo <= score <= hi:
            return label
    return 'unknown'

print(f'Test set: {len(test_records)} records')
print(f'Keys: {list(test_records[0].keys())}')

Test set: 2000 records
Keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


In [3]:
# === CELL 3: Load 5 models từ Drive, tính ensemble predictions ===
from google.colab import drive
drive.mount('/content/drive')

drive_base = '/content/drive/MyDrive/ai-recruiter/models'
seeds = [42, 123, 456, 789, 999]
sentence_pairs = [[r['cv_text'], r['jd_text']] for r in test_records]

all_preds = []
for run_idx, seed in enumerate(seeds, 1):
    run_name = f'v0.6-ensemble-run{run_idx}-seed{seed}'
    model_path = f'{drive_base}/{run_name}'
    if not os.path.exists(model_path):
        print(f'MISSING: {run_name}')
        continue
    model = CrossEncoder(model_path)
    preds = model.predict(sentence_pairs, batch_size=32, show_progress_bar=False)
    all_preds.append(preds)
    print(f'Run {run_idx} (seed={seed}): loaded')

# Ensemble = average
ensemble_preds_01 = np.mean(all_preds, axis=0)          # 0-1 scale
ensemble_preds    = ensemble_preds_01 * 100              # 0-100 scale
true_scores       = np.array([r['score'] for r in test_records])
errors            = ensemble_preds - true_scores         # signed error
abs_errors        = np.abs(errors)

print(f'\nEnsemble LabelAcc (±10): {np.mean(abs_errors <= 10)*100:.2f}%')
print(f'Mean error (bias):        {np.mean(errors):+.2f} points')

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Run 1 (seed=42): loaded


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Run 2 (seed=123): loaded


...

Run 3 (seed=456): loaded


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Run 4 (seed=789): loaded


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Run 5 (seed=999): loaded

Ensemble LabelAcc (±10): 65.80%
Mean error (bias):        +0.45 points


In [4]:
# === CELL 4: Per-class accuracy và bias ===
from collections import defaultdict

class_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'errors': [], 'signed_errors': []})

for i, rec in enumerate(test_records):
    true_label = rec['true_label']
    pred_score = ensemble_preds[i]
    true_score = rec['score']
    err = abs_errors[i]
    signed_err = errors[i]

    class_stats[true_label]['total'] += 1
    class_stats[true_label]['errors'].append(err)
    class_stats[true_label]['signed_errors'].append(signed_err)
    if err <= 10:
        class_stats[true_label]['correct'] += 1

print('=== Per-class accuracy & bias ===')
print(f'{"Class":<20} {"Acc":<10} {"Bias":<12} {"MAE":<10} {"Count"}')
print('-' * 62)
for label in LABELS_ORDER:
    s = class_stats[label]
    if s['total'] == 0:
        continue
    acc  = s['correct'] / s['total'] * 100
    bias = np.mean(s['signed_errors'])   # positive = over-predict, negative = under-predict
    mae  = np.mean(s['errors'])
    print(f'{label:<20} {acc:>7.1f}%  {bias:>+8.1f} pts  {mae:>7.1f}  {s["total"]}')

=== Per-class accuracy & bias ===
Class                Acc        Bias         MAE        Count
--------------------------------------------------------------
poor_match              60.2%      +7.8 pts     10.1  410
weak_match              52.3%      +1.6 pts     12.6  392
moderate_match          63.4%      -1.0 pts      9.4  388
strong_match            71.4%      -2.6 pts      8.3  377
excellent_match         80.6%      -3.6 pts      5.4  433


In [5]:
# === CELL 5: Confusion matrix (true label → predicted label) ===
import pandas as pd

confusion = defaultdict(lambda: defaultdict(int))

for i, rec in enumerate(test_records):
    true_label = rec['true_label']
    pred_label = score_to_label(round(ensemble_preds[i]))
    confusion[true_label][pred_label] += 1

# Print as matrix
print('=== Confusion Matrix (rows=true, cols=predicted) ===')
header = f'{"":20}' + ''.join(f'{l[:8]:>12}' for l in LABELS_ORDER)
print(header)
print('-' * (20 + 12 * len(LABELS_ORDER)))
for true_label in LABELS_ORDER:
    row = f'{true_label:<20}'
    total = sum(confusion[true_label].values())
    for pred_label in LABELS_ORDER:
        n = confusion[true_label][pred_label]
        pct = n / total * 100 if total > 0 else 0
        marker = '<<' if true_label == pred_label else '  '
        row += f'{n:>6}({pct:4.0f}%){marker}'
    print(row)

print('\n(<<) = diagonal = correct predictions')

=== Confusion Matrix (rows=true, cols=predicted) ===
                        poor_mat    weak_mat    moderate    strong_m    excellen
--------------------------------------------------------------------------------
poor_match             305(  74%)<<    93(  23%)      12(   3%)       0(   0%)       0(   0%)  
weak_match              91(  23%)     193(  49%)<<    74(  19%)      28(   7%)       6(   2%)  
moderate_match          13(   3%)      80(  21%)     206(  53%)<<    75(  19%)      14(   4%)  
strong_match             3(   1%)      24(   6%)      66(  18%)     211(  56%)<<    71(  19%)  
excellent_match          1(   0%)       1(   0%)      14(   3%)      86(  20%)     319(  74%)<<

(<<) = diagonal = correct predictions


In [6]:
# === CELL 6: Error magnitude distribution ===
bins = [(0, 5), (5, 10), (10, 15), (15, 20), (20, 30), (30, 100)]
labels_desc = ['0-5 (great)', '5-10 (ok)', '10-15 (miss)', '15-20 (bad)', '20-30 (very bad)', '>30 (terrible)']

print('=== Error magnitude distribution ===')
print(f'{"Range":<20} {"Count":>8} {"Pct":>8}')
print('-' * 40)
for (lo, hi), desc in zip(bins, labels_desc):
    count = np.sum((abs_errors >= lo) & (abs_errors < hi))
    pct = count / len(abs_errors) * 100
    bar = '█' * int(pct / 2)
    print(f'{desc:<20} {count:>8} {pct:>7.1f}%  {bar}')

print(f'\nTotal wrong (>10): {np.sum(abs_errors > 10)} / {len(abs_errors)} ({np.mean(abs_errors > 10)*100:.1f}%)')

=== Error magnitude distribution ===
Range                   Count      Pct
----------------------------------------
0-5 (great)               887    44.4%  ██████████████████████
5-10 (ok)                 429    21.4%  ██████████
10-15 (miss)              266    13.3%  ██████
15-20 (bad)               175     8.8%  ████
20-30 (very bad)          162     8.1%  ████
>30 (terrible)             81     4.0%  ██

Total wrong (>10): 684 / 2000 (34.2%)


In [7]:
# === CELL 7: Systematic bias — model có xu hướng over/under-predict ở range nào? ===
print('=== Signed error by true score range (positive = over-predict) ===')
score_bins = [(0, 20), (20, 40), (40, 60), (60, 75), (75, 90), (90, 101)]
score_labels = ['0-19 (poor)', '20-39 (poor)', '40-59 (weak)', '60-74 (mod)', '75-89 (strong)', '90-100 (excel)']

print(f'{"Range":<20} {"Count":>6} {"Mean error":>12} {"Std":>8}')
print('-' * 50)
for (lo, hi), desc in zip(score_bins, score_labels):
    mask = (true_scores >= lo) & (true_scores < hi)
    if mask.sum() == 0:
        continue
    mean_err = np.mean(errors[mask])
    std_err  = np.std(errors[mask])
    direction = '↑ over' if mean_err > 2 else ('↓ under' if mean_err < -2 else '≈ ok')
    print(f'{desc:<20} {mask.sum():>6} {mean_err:>+10.1f} {std_err:>8.1f}  {direction}')

=== Signed error by true score range (positive = over-predict) ===
Range                 Count   Mean error      Std
--------------------------------------------------
0-19 (poor)              79       +8.4      7.9  ↑ over
20-39 (poor)            331       +7.6     12.1  ↑ over
40-59 (weak)            392       +1.6     16.1  ≈ ok
60-74 (mod)             388       -1.0     12.5  ≈ ok
75-89 (strong)          377       -2.6     11.4  ↓ under
90-100 (excel)          433       -3.6      8.7  ↓ under


In [8]:
# === CELL 8: Worst-case examples — top 20 pairs bị predict sai nhiều nhất ===
top_errors_idx = np.argsort(abs_errors)[::-1][:20]

print('=== Top 20 worst predictions ===')
print(f'{"#":<4} {"True":>6} {"Pred":>6} {"Err":>6} {"True label":<20} {"Pred label":<20}')
print('-' * 70)
for rank, idx in enumerate(top_errors_idx, 1):
    rec        = test_records[idx]
    true_score = rec['score']
    pred_score = ensemble_preds[idx]
    true_lbl   = rec['true_label']
    pred_lbl   = score_to_label(round(pred_score))
    err        = abs_errors[idx]
    print(f'{rank:<4} {true_score:>6.0f} {pred_score:>6.1f} {err:>6.1f} {true_lbl:<20} {pred_lbl:<20}')

print('\n=== CV/JD snippets for worst 3 ===')
for rank, idx in enumerate(top_errors_idx[:3], 1):
    rec = test_records[idx]
    print(f'\n--- #{rank} (true={rec["score"]:.0f}, pred={ensemble_preds[idx]:.1f}, err={abs_errors[idx]:.1f}) ---')
    print(f'True label: {rec["true_label"]}')
    print(f'CV:  {rec["cv_text"][:300]}')
    print(f'JD:  {rec["jd_text"][:300]}')

=== Top 20 worst predictions ===
#      True   Pred    Err True label           Pred label          
----------------------------------------------------------------------
1        99   21.1   77.9 excellent_match      poor_match          
2        84   34.4   49.6 strong_match         poor_match          
3        49   97.1   48.1 weak_match           excellent_match     
4        25   71.8   46.8 poor_match           moderate_match      
5        81   34.8   46.2 strong_match         poor_match          
6        52   97.9   45.9 weak_match           excellent_match     
7        53   98.5   45.5 weak_match           excellent_match     
8        27   71.5   44.5 poor_match           moderate_match      
9        30   72.8   42.8 poor_match           moderate_match      
10       75   34.1   40.9 strong_match         poor_match          
11       48   88.7   40.7 weak_match           strong_match        
12       24   63.8   39.8 poor_match           moderate_match      
13       53 

In [9]:
# === CELL 9: Save full error analysis report ===
import os
os.makedirs('artifacts/reports', exist_ok=True)

# Build per-record results
per_record = []
for i, rec in enumerate(test_records):
    pred_score = float(ensemble_preds[i])
    per_record.append({
        'pair_id':      rec.get('pair_id', i),
        'true_score':   rec['score'],
        'pred_score':   round(pred_score, 2),
        'true_label':   rec['true_label'],
        'pred_label':   score_to_label(round(pred_score)),
        'abs_error':    round(float(abs_errors[i]), 2),
        'signed_error': round(float(errors[i]), 2),
        'correct':      bool(abs_errors[i] <= 10),
    })

# Summary stats
summary = {
    'overall': {
        'labelacc': round(float(np.mean(abs_errors <= 10)), 4),
        'mae':      round(float(np.mean(abs_errors)), 4),
        'bias':     round(float(np.mean(errors)), 4),
    },
    'per_class': {},
    'confusion_matrix': {},
    'error_magnitude': {},
}

for label in LABELS_ORDER:
    s = class_stats[label]
    if s['total'] == 0:
        continue
    summary['per_class'][label] = {
        'labelacc': round(s['correct'] / s['total'], 4),
        'mae':      round(float(np.mean(s['errors'])), 4),
        'bias':     round(float(np.mean(s['signed_errors'])), 4),
        'count':    s['total'],
    }

for true_label in LABELS_ORDER:
    summary['confusion_matrix'][true_label] = dict(confusion[true_label])

for (lo, hi), desc in zip(bins, labels_desc):
    count = int(np.sum((abs_errors >= lo) & (abs_errors < hi)))
    summary['error_magnitude'][desc] = {'count': count, 'pct': round(count / len(abs_errors) * 100, 1)}

report = {
    'model':   'cross-encoder-v0.6-ensemble-5seeds',
    'dataset': 'v0.5/cross_encoder/test (2000 pairs)',
    'summary': summary,
    'per_record': per_record,
}

report_path = 'artifacts/reports/error_analysis_cross_encoder_v0.6_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f'Report saved: {report_path}')
print(f'Total records: {len(per_record)}')

# Also copy to Drive
try:
    import shutil
    drive_reports = '/content/drive/MyDrive/ai-recruiter/reports'
    os.makedirs(drive_reports, exist_ok=True)
    shutil.copy(report_path, f'{drive_reports}/error_analysis_cross_encoder_v0.6_report.json')
    print('Copied to Drive')
except Exception as e:
    print(f'Drive copy skipped: {e}')

Report saved: artifacts/reports/error_analysis_cross_encoder_v0.6_report.json
Total records: 2000
Copied to Drive
